In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🌾 OPTICROP - DATA PREPROCESSING")
print("="*60)

🌾 OPTICROP - DATA PREPROCESSING


In [2]:
# Load dataset
print("\n📂 1. Loading Dataset...")

# Try multiple paths
paths = [
    '../../../Dataset/Crop_recommendation.csv',
    '../../Dataset/Crop_recommendation.csv',
    '../Dataset/Crop_recommendation.csv',
    'Dataset/Crop_recommendation.csv'
]

df = None
for path in paths:
    try:
        df = pd.read_csv(path)
        print(f"   ✅ Loaded from: {path}")
        break
    except FileNotFoundError:
        continue

if df is None:
    print("   ❌ Error: Dataset not found!")
    print("   ⚠️ Please make sure the dataset exists")
else:
    print(f"   ✅ Loaded {len(df)} rows and {len(df.columns)} columns")
    print(f"   📋 Columns: {list(df.columns)}")


📂 1. Loading Dataset...
   ✅ Loaded from: ../../Dataset/Crop_recommendation.csv
   ✅ Loaded 2200 rows and 8 columns
   📋 Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']


In [3]:
# Data overview
print("\n📊 2. Data Overview...")
print(f"   Shape: {df.shape}")
print(f"   Features: {list(df.columns)}")


📊 2. Data Overview...
   Shape: (2200, 8)
   Features: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']


In [4]:
# Check missing values
print("\n🔍 3. Checking for Missing Values...")
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("   ✅ No missing values found!")
else:
    print(f"   ⚠️ Found {missing_values.sum()} missing values:")
    print(missing_values[missing_values > 0])


🔍 3. Checking for Missing Values...
   ✅ No missing values found!


In [5]:
# Data types
print("\n📋 4. Data Types...")
print(df.dtypes)


📋 4. Data Types...
N                int64
P                int64
K                int64
temperature    float64
humidity       float64
ph             float64
rainfall       float64
label           object
dtype: object


In [6]:
# Handle outliers using IQR method
print("\n📊 5. Handling Outliers...")

numeric_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
outliers_count = 0

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outliers_count += len(outliers)
    
    if len(outliers) > 0:
        print(f"   {col}: {len(outliers)} outliers detected")
        # Cap outliers
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print(f"   ✅ Total outliers handled: {outliers_count}")


📊 5. Handling Outliers...
   P: 138 outliers detected
   K: 200 outliers detected
   temperature: 86 outliers detected
   humidity: 30 outliers detected
   ph: 57 outliers detected
   rainfall: 100 outliers detected
   ✅ Total outliers handled: 611


In [7]:
# Crop distribution
print("\n🌾 6. Crop Distribution...")
crop_counts = df['label'].value_counts()
print(f"   Total unique crops: {len(crop_counts)}")
print("   Crop distribution:")
for crop, count in crop_counts.items():
    print(f"      - {crop}: {count} samples")


🌾 6. Crop Distribution...
   Total unique crops: 22
   Crop distribution:
      - rice: 100 samples
      - maize: 100 samples
      - jute: 100 samples
      - cotton: 100 samples
      - coconut: 100 samples
      - papaya: 100 samples
      - orange: 100 samples
      - apple: 100 samples
      - muskmelon: 100 samples
      - watermelon: 100 samples
      - grapes: 100 samples
      - mango: 100 samples
      - banana: 100 samples
      - pomegranate: 100 samples
      - lentil: 100 samples
      - blackgram: 100 samples
      - mungbean: 100 samples
      - mothbeans: 100 samples
      - pigeonpeas: 100 samples
      - kidneybeans: 100 samples
      - chickpea: 100 samples
      - coffee: 100 samples


In [8]:
# Encode target variable
print("\n🔢 7. Encoding Target Variable...")

label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])
print(f"   ✅ Encoded {len(label_encoder.classes_)} unique crops")
print("   📋 Crop mapping:")
for i, crop in enumerate(label_encoder.classes_):
    print(f"      - {crop}: {i}")


🔢 7. Encoding Target Variable...
   ✅ Encoded 22 unique crops
   📋 Crop mapping:
      - apple: 0
      - banana: 1
      - blackgram: 2
      - chickpea: 3
      - coconut: 4
      - coffee: 5
      - cotton: 6
      - grapes: 7
      - jute: 8
      - kidneybeans: 9
      - lentil: 10
      - maize: 11
      - mango: 12
      - mothbeans: 13
      - mungbean: 14
      - muskmelon: 15
      - orange: 16
      - papaya: 17
      - pigeonpeas: 18
      - pomegranate: 19
      - rice: 20
      - watermelon: 21


In [9]:
# Feature scaling
print("\n📊 8. Feature Scaling...")

# Separate features and target
numeric_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[numeric_cols].values
y = df['label_encoded'].values

# Initialize scaler
scaler = StandardScaler()

# Fit and transform features
X_scaled = scaler.fit_transform(X)
print(f"   ✅ Features scaled successfully!")


📊 8. Feature Scaling...
   ✅ Features scaled successfully!


In [10]:
# Train-test split
print("\n📊 9. Train-Test Split...")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   ✅ Training samples: {len(X_train)} (80%)")
print(f"   ✅ Testing samples: {len(X_test)} (20%)")
print(f"   📊 Training crops: {len(set(y_train))} unique")
print(f"   📊 Testing crops: {len(set(y_test))} unique")


📊 9. Train-Test Split...
   ✅ Training samples: 1760 (80%)
   ✅ Testing samples: 440 (20%)
   📊 Training crops: 22 unique
   📊 Testing crops: 22 unique


In [11]:
# Save processed data
print("\n💾 10. Saving Processed Data...")

# Create processed folder if not exists
os.makedirs('processed', exist_ok=True)

# Save processed dataset
processed_df = df.copy()
processed_df['N_scaled'] = X_scaled[:, 0]
processed_df['P_scaled'] = X_scaled[:, 1]
processed_df['K_scaled'] = X_scaled[:, 2]
processed_df['temp_scaled'] = X_scaled[:, 3]
processed_df['humidity_scaled'] = X_scaled[:, 4]
processed_df['ph_scaled'] = X_scaled[:, 5]
processed_df['rainfall_scaled'] = X_scaled[:, 6]

processed_df.to_csv('processed/processed_crop_data.csv', index=False)
print("   ✅ Saved processed dataset to 'processed/processed_crop_data.csv'")

# Save scaler
with open('processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("   ✅ Saved scaler to 'processed/scaler.pkl'")

# Save label encoder
with open('processed/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("   ✅ Saved label encoder to 'processed/label_encoder.pkl'")

# Save train-test split
np.save('processed/X_train.npy', X_train)
np.save('processed/X_test.npy', X_test)
np.save('processed/y_train.npy', y_train)
np.save('processed/y_test.npy', y_test)
print("   ✅ Saved train-test split to 'processed/'")


💾 10. Saving Processed Data...
   ✅ Saved processed dataset to 'processed/processed_crop_data.csv'
   ✅ Saved scaler to 'processed/scaler.pkl'
   ✅ Saved label encoder to 'processed/label_encoder.pkl'
   ✅ Saved train-test split to 'processed/'


In [12]:
# Completion
print("\n" + "="*60)
print("✅ PREPROCESSING COMPLETED SUCCESSFULLY!")
print("📁 All processed files saved to 'Preprocessing/processed/'")
print("="*60)

print("\n📦 Processed data ready for model training!")
print(f"   🎯 Features: {len(numeric_cols)}")
print(f"   🌾 Crops: {len(label_encoder.classes_)}")
print(f"   📊 Training samples: {len(X_train)}")
print(f"   📊 Testing samples: {len(X_test)}")


✅ PREPROCESSING COMPLETED SUCCESSFULLY!
📁 All processed files saved to 'Preprocessing/processed/'

📦 Processed data ready for model training!
   🎯 Features: 7
   🌾 Crops: 22
   📊 Training samples: 1760
   📊 Testing samples: 440
